# Task 1: Convolutional-Recurrent Architecture for Artwork Classification

**ArtExtract — GSoC 2025 Evaluation Task**

---

## Overview

This notebook implements a pipeline for classifying paintings by **Style**, **Artist**, and **Genre** using a **Convolutional-Recurrent (CNN-LSTM)** architecture, trained on the [WikiArt / ArtGAN dataset](https://github.com/cs-chan/ArtGAN/blob/master/WikiArt%20Dataset/README.md).

### Strategy

Paintings are rich visual objects that contain both **local texture patterns** (brushstrokes, colors, composition details) and **global structural cues** (pose of figures, spatial arrangement). A pure CNN captures local features well, but a recurrent layer on top allows the model to integrate spatial context across different regions of the image (treating a grid of CNN-extracted patch features as a sequence).

The chosen approach is:
1. **CNN Backbone** — EfficientNet-B3 pre-trained on ImageNet, used as a feature extractor.
2. **Sequence formation** — The spatial feature map (H×W grid of feature vectors) is flattened into a sequence of patch embeddings.
3. **Bidirectional LSTM** — Captures long-range dependencies between image regions.
4. **Multi-head classifier** — Three separate classification heads for Style, Artist, and Genre.
5. **Outlier detection** — Using reconstruction error from an autoencoder branch and t-SNE / UMAP visualizations to surface paintings that are misassigned.

### Evaluation Metrics
- **Top-1 / Top-5 Accuracy** — Standard classification accuracy.
- **Macro F1-Score** — Accounts for class imbalance, which is severe in the WikiArt dataset.
- **Confusion Matrix** — To identify which styles/artists are most often confused.
- **t-SNE / UMAP embedding plots** — Qualitative check of cluster separation.
- **Outlier score** — Cosine distance from each sample to its predicted-class centroid in the embedding space.

---
## 1. Environment Setup & Library Imports

In [ ]:
# ── Core scientific stack ────────────────────────────────────────────────────
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

# ── PyTorch & torchvision ────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

# ── Sklearn utilities ────────────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder

# ── Image loading ────────────────────────────────────────────────────────────
from PIL import Image

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

---
## 2. Dataset: WikiArt / ArtGAN

The [ArtGAN WikiArt dataset](https://github.com/cs-chan/ArtGAN/blob/master/WikiArt%20Dataset/README.md) contains **~80 000 paintings** across:
- **27 Art Styles** (Impressionism, Cubism, Baroque, …)
- **195 Artists** (Monet, Picasso, Rembrandt, …)
- **10 Genres** (portrait, landscape, religious painting, …)

The dataset provides CSV split files (`train.csv`, `val.csv`, `test.csv`) with columns:
`image_path, style_label, artist_label, genre_label`.

Below we define paths and a helper to inspect class distributions.

In [ ]:
# ── Dataset paths ─────────────────────────────────────────────────────────────
# Update DATA_ROOT to the local path where you extracted the WikiArt dataset.
DATA_ROOT = Path("data/wikiart")
TRAIN_CSV = DATA_ROOT / "train.csv"
VAL_CSV   = DATA_ROOT / "val.csv"
TEST_CSV  = DATA_ROOT / "test.csv"

# Label columns
STYLE_COL  = "style_label"
ARTIST_COL = "artist_label"
GENRE_COL  = "genre_label"
IMG_COL    = "image_path"

# Hyper-parameters
IMG_SIZE    = 224          # resize shorter side to IMG_SIZE
BATCH_SIZE  = 32
NUM_EPOCHS  = 20
LR          = 1e-4
WEIGHT_DECAY = 1e-5
LSTM_HIDDEN = 512
LSTM_LAYERS = 2

print("Paths configured.  Update DATA_ROOT if needed.")

In [ ]:
def load_csv_or_demo(csv_path: Path, demo_rows: int = 200) -> pd.DataFrame:
    """Load split CSV, or return a tiny synthetic demo DataFrame."""
    if csv_path.exists():
        return pd.read_csv(csv_path)
    # ── Demo mode: build a fake DataFrame so the notebook is runnable ─────
    print(f"[DEMO] {csv_path} not found — generating synthetic data.")
    rng = np.random.default_rng(SEED)
    styles  = [f"style_{i}"  for i in range(10)]
    artists = [f"artist_{i}" for i in range(20)]
    genres  = [f"genre_{i}"  for i in range(5)]
    return pd.DataFrame({
        IMG_COL:    [f"img_{i:05d}.jpg" for i in range(demo_rows)],
        STYLE_COL:  rng.choice(styles,  demo_rows),
        ARTIST_COL: rng.choice(artists, demo_rows),
        GENRE_COL:  rng.choice(genres,  demo_rows),
    })


train_df = load_csv_or_demo(TRAIN_CSV)
val_df   = load_csv_or_demo(VAL_CSV,  demo_rows=50)
test_df  = load_csv_or_demo(TEST_CSV, demo_rows=50)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
train_df.head(3)

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col, title in zip(axes,
                          [STYLE_COL, ARTIST_COL, GENRE_COL],
                          ["Style", "Artist", "Genre"]):
    counts = train_df[col].value_counts().head(15)
    counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(f"Top-15 {title} distribution (train)")
    ax.set_xlabel(title)
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

---
## 3. Data Preprocessing & Augmentation

In [ ]:
# ── Label encoding ────────────────────────────────────────────────────────────
le_style  = LabelEncoder().fit(train_df[STYLE_COL])
le_artist = LabelEncoder().fit(train_df[ARTIST_COL])
le_genre  = LabelEncoder().fit(train_df[GENRE_COL])

NUM_STYLES  = len(le_style.classes_)
NUM_ARTISTS = len(le_artist.classes_)
NUM_GENRES  = len(le_genre.classes_)
print(f"Unique classes — Style: {NUM_STYLES}, Artist: {NUM_ARTISTS}, Genre: {NUM_GENRES}")

# ── Transforms ───────────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Transforms defined.")

In [ ]:
class WikiArtDataset(Dataset):
    """PyTorch Dataset for WikiArt paintings.

    Parameters
    ----------
    df         : DataFrame with columns [IMG_COL, STYLE_COL, ARTIST_COL, GENRE_COL]
    root       : Root directory where images are stored
    transform  : torchvision transforms to apply
    le_style / le_artist / le_genre : fitted LabelEncoders
    demo_mode  : if True, returns a random tensor instead of reading disk images
    """

    def __init__(self, df, root, transform,
                 le_style, le_artist, le_genre, demo_mode=False):
        self.df        = df.reset_index(drop=True)
        self.root      = Path(root)
        self.transform = transform
        self.le_style  = le_style
        self.le_artist = le_artist
        self.le_genre  = le_genre
        self.demo_mode = demo_mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        if self.demo_mode:
            # Return a random tensor so the pipeline can be tested without images
            image = torch.randn(3, IMG_SIZE, IMG_SIZE)
        else:
            img_path = self.root / row[IMG_COL]
            image = Image.open(img_path).convert("RGB")
            image = self.transform(image)

        style_label  = int(self.le_style.transform([row[STYLE_COL]])[0])
        artist_label = int(self.le_artist.transform([row[ARTIST_COL]])[0])
        genre_label  = int(self.le_genre.transform([row[GENRE_COL]])[0])

        return image, style_label, artist_label, genre_label


# Detect whether we're in demo mode
DEMO = not DATA_ROOT.exists()

train_dataset = WikiArtDataset(train_df, DATA_ROOT, train_transforms,
                               le_style, le_artist, le_genre, demo_mode=DEMO)
val_dataset   = WikiArtDataset(val_df,   DATA_ROOT, eval_transforms,
                               le_style, le_artist, le_genre, demo_mode=DEMO)
test_dataset  = WikiArtDataset(test_df,  DATA_ROOT, eval_transforms,
                               le_style, le_artist, le_genre, demo_mode=DEMO)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"DataLoaders ready.  Demo mode: {DEMO}")

---
## 4. Model Architecture — CNN + Bidirectional LSTM

```
Input image (3 × 224 × 224)
       │
  EfficientNet-B3 backbone
  (remove classification head)     →  feature map  (1536 × 7 × 7)
       │
  Reshape to sequence              →  (49 patches × 1536 features)
       │
  Bidirectional LSTM (2 layers)    →  (49 × 1024)  last hidden: (1024,)
       │
  ┌────┴────┬────────┐
 Style   Artist   Genre
 head     head    head
```

Each classification head is a two-layer MLP with dropout and batch-norm.

In [ ]:
class ClassificationHead(nn.Module):
    """A small MLP head: Linear → BN → ReLU → Dropout → Linear."""

    def __init__(self, in_features: int, num_classes: int, dropout: float = 0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class CNNLSTMArtClassifier(nn.Module):
    """
    CNN-LSTM multi-task classifier for artwork attributes.

    Parameters
    ----------
    num_styles, num_artists, num_genres : number of output classes per task
    lstm_hidden : hidden size of the Bi-LSTM
    lstm_layers : number of stacked LSTM layers
    freeze_backbone : if True, freeze CNN weights during early training
    """

    def __init__(self,
                 num_styles: int,
                 num_artists: int,
                 num_genres: int,
                 lstm_hidden: int = 512,
                 lstm_layers: int = 2,
                 freeze_backbone: bool = True):
        super().__init__()

        # ── CNN backbone ────────────────────────────────────────────────────
        backbone = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
        # Remove the classifier; keep only the feature extractor
        self.cnn = backbone.features          # output: (B, 1536, 7, 7)
        self.cnn_out_channels = 1536
        self.pool_h = self.pool_w = 7         # spatial size after backbone

        if freeze_backbone:
            for p in self.cnn.parameters():
                p.requires_grad = False

        # ── Bi-LSTM ─────────────────────────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size=self.cnn_out_channels,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3 if lstm_layers > 1 else 0.0,
        )
        # Bidirectional → 2 × lstm_hidden
        self.lstm_out_dim = lstm_hidden * 2

        # ── Attention over LSTM timesteps ───────────────────────────────────
        self.attention = nn.Linear(self.lstm_out_dim, 1)

        # ── Classification heads ─────────────────────────────────────────────
        self.style_head  = ClassificationHead(self.lstm_out_dim, num_styles)
        self.artist_head = ClassificationHead(self.lstm_out_dim, num_artists)
        self.genre_head  = ClassificationHead(self.lstm_out_dim, num_genres)

    def forward(self, x):
        B = x.size(0)

        # 1. CNN feature extraction → (B, C, H, W)
        feat = self.cnn(x)                       # (B, 1536, 7, 7)

        # 2. Reshape spatial grid into a sequence → (B, H*W, C)
        feat = feat.flatten(2).permute(0, 2, 1)  # (B, 49, 1536)

        # 3. Bi-LSTM over patch sequence → (B, 49, lstm_out_dim)
        lstm_out, _ = self.lstm(feat)

        # 4. Attention pooling → (B, lstm_out_dim)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)  # (B, 49, 1)
        context = (attn_weights * lstm_out).sum(dim=1)                 # (B, lstm_out_dim)

        # 5. Classification heads
        style_logits  = self.style_head(context)
        artist_logits = self.artist_head(context)
        genre_logits  = self.genre_head(context)

        return style_logits, artist_logits, genre_logits

    def get_embedding(self, x):
        """Return the attention-pooled context vector (useful for outlier detection)."""
        B = x.size(0)
        feat = self.cnn(x).flatten(2).permute(0, 2, 1)
        lstm_out, _ = self.lstm(feat)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = (attn_weights * lstm_out).sum(dim=1)
        return context


# ── Instantiate and inspect ───────────────────────────────────────────────────
model = CNNLSTMArtClassifier(
    num_styles=NUM_STYLES,
    num_artists=NUM_ARTISTS,
    num_genres=NUM_GENRES,
    lstm_hidden=LSTM_HIDDEN,
    lstm_layers=LSTM_LAYERS,
    freeze_backbone=True,
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# ── Quick forward-pass smoke test ─────────────────────────────────────────────
dummy_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    s_out, a_out, g_out = model(dummy_input)
print(f"Style  logits shape : {s_out.shape}")   # expected: (2, NUM_STYLES)
print(f"Artist logits shape : {a_out.shape}")   # expected: (2, NUM_ARTISTS)
print(f"Genre  logits shape : {g_out.shape}")   # expected: (2, NUM_GENRES)

---
## 5. Loss Function, Optimizer & Scheduler

In [ ]:
# ── Multi-task loss with learnable task weights (uncertainty weighting) ────────
# Reference: Kendall et al., "Multi-Task Learning Using Uncertainty to Weigh
# Losses for Scene Geometry and Semantics" (CVPR 2018)
class MultiTaskLoss(nn.Module):
    """Homoscedastic uncertainty weighting for multi-task classification."""

    def __init__(self, n_tasks: int = 3):
        super().__init__()
        # log(sigma^2) for each task — learnable
        self.log_sigma_sq = nn.Parameter(torch.zeros(n_tasks))
        self.ce = nn.CrossEntropyLoss()

    def forward(self, preds, targets):
        """
        preds   : list of (logits_style, logits_artist, logits_genre)
        targets : list of (style_label, artist_label, genre_label)
        """
        losses = [self.ce(p, t) for p, t in zip(preds, targets)]
        total = sum(
            torch.exp(-self.log_sigma_sq[i]) * losses[i]
            + 0.5 * self.log_sigma_sq[i]
            for i in range(len(losses))
        )
        return total, losses


criterion = MultiTaskLoss(n_tasks=3).to(DEVICE)

# ── Optimizer: AdamW on trainable params + task-weight params ─────────────────
optimizer = optim.AdamW(
    list(filter(lambda p: p.requires_grad, model.parameters()))
    + list(criterion.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

# ── Cosine annealing with warm restarts ───────────────────────────────────────
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=5, T_mult=2, eta_min=1e-6
)

print("Loss, optimizer, and scheduler configured.")

---
## 6. Training Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = {"style": 0, "artist": 0, "genre": 0}
    total = 0

    for images, s_lbls, a_lbls, g_lbls in tqdm(loader, leave=False, desc="train"):
        images = images.to(device)
        s_lbls = s_lbls.to(device)
        a_lbls = a_lbls.to(device)
        g_lbls = g_lbls.to(device)

        optimizer.zero_grad()
        s_out, a_out, g_out = model(images)
        loss, _ = criterion([s_out, a_out, g_out], [s_lbls, a_lbls, g_lbls])
        loss.backward()
        trainable = [p for p in model.parameters() if p.requires_grad]
        nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total += images.size(0)
        correct["style"]  += (s_out.argmax(1) == s_lbls).sum().item()
        correct["artist"] += (a_out.argmax(1) == a_lbls).sum().item()
        correct["genre"]  += (g_out.argmax(1) == g_lbls).sum().item()

    epoch_loss = running_loss / total
    accs = {k: v / total for k, v in correct.items()}
    return epoch_loss, accs


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds  = {"style": [], "artist": [], "genre": []}
    all_labels = {"style": [], "artist": [], "genre": []}
    total = 0

    for images, s_lbls, a_lbls, g_lbls in tqdm(loader, leave=False, desc="eval "):
        images = images.to(device)
        s_lbls = s_lbls.to(device)
        a_lbls = a_lbls.to(device)
        g_lbls = g_lbls.to(device)

        s_out, a_out, g_out = model(images)
        loss, _ = criterion([s_out, a_out, g_out], [s_lbls, a_lbls, g_lbls])

        running_loss += loss.item() * images.size(0)
        total += images.size(0)

        for key, out, lbls in [("style", s_out, s_lbls),
                                ("artist", a_out, a_lbls),
                                ("genre",  g_out, g_lbls)]:
            all_preds[key].extend(out.argmax(1).cpu().numpy())
            all_labels[key].extend(lbls.cpu().numpy())

    epoch_loss = running_loss / total
    f1s = {
        k: f1_score(all_labels[k], all_preds[k], average="macro", zero_division=0)
        for k in all_preds
    }
    accs = {
        k: accuracy_score(all_labels[k], all_preds[k])
        for k in all_preds
    }
    return epoch_loss, accs, f1s, all_preds, all_labels


print("Training & evaluation functions defined.")

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
# NOTE: Set NUM_EPOCHS_RUN to a small number for a quick demo; use NUM_EPOCHS for full training.
NUM_EPOCHS_RUN = 2  # change to NUM_EPOCHS for full training

history = {"train_loss": [], "val_loss": [],
           "val_acc_style": [], "val_acc_artist": [], "val_acc_genre": [],
           "val_f1_style":  [], "val_f1_artist":  [], "val_f1_genre":  []}

best_val_loss = float("inf")
CKPT_PATH = Path("checkpoints/best_model.pth")
CKPT_PATH.parent.mkdir(exist_ok=True)

for epoch in range(1, NUM_EPOCHS_RUN + 1):
    tr_loss, tr_accs = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_accs, val_f1s, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step(epoch)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(val_loss)
    for task in ["style", "artist", "genre"]:
        history[f"val_acc_{task}"].append(val_accs[task])
        history[f"val_f1_{task}"].append(val_f1s[task])

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CKPT_PATH)

    print(f"Epoch {epoch:>3}/{NUM_EPOCHS_RUN} | "
          f"Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Style Acc: {val_accs['style']:.3f} | "
          f"Artist Acc: {val_accs['artist']:.3f} | "
          f"Genre Acc: {val_accs['genre']:.3f}")

print("\nTraining complete.")

---
## 7. Training Curves

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(epochs, history["train_loss"], label="Train Loss", marker="o")
axes[0].plot(epochs, history["val_loss"],   label="Val Loss",   marker="s")
axes[0].set_title("Loss curves")
axes[0].set_xlabel("Epoch")
axes[0].legend()

for task, color in [("style", "tab:blue"), ("artist", "tab:orange"), ("genre", "tab:green")]:
    axes[1].plot(epochs, history[f"val_f1_{task}"], label=f"{task.capitalize()} F1",
                 marker="o", color=color)
axes[1].set_title("Validation Macro-F1 per task")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 8. Evaluation on Test Set

In [ ]:
# Load best checkpoint and evaluate on test set
if CKPT_PATH.exists():
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))

test_loss, test_accs, test_f1s, test_preds, test_labels = evaluate(
    model, test_loader, criterion, DEVICE
)

print("\n── Test Results ──────────────────────────────────")
for task in ["style", "artist", "genre"]:
    print(f"  {task.capitalize():8s} | Acc: {test_accs[task]:.4f} | Macro-F1: {test_f1s[task]:.4f}")

In [ ]:
# ── Confusion matrix for Style classification ─────────────────────────────────
cm_style = confusion_matrix(test_labels["style"], test_preds["style"])

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_style, annot=False, fmt="d", cmap="Blues",
            xticklabels=le_style.classes_,
            yticklabels=le_style.classes_, ax=ax)
ax.set_title("Confusion Matrix — Style Classification")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("\nClassification Report (Style):")
print(classification_report(test_labels["style"], test_preds["style"],
                             target_names=le_style.classes_, zero_division=0))

---
## 9. Outlier Detection

We identify **outlier paintings** — images that are statistically far from the centroid of their assigned label in the embedding space.  These often correspond to misattributed works, atypical pieces, or labelling errors.

**Method:**  
1. Extract CNN-LSTM embeddings for all test images.  
2. Compute the per-class centroid in embedding space.  
3. Rank images by cosine distance to their class centroid.  
4. Top-ranked images are candidate outliers.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader, device):
    model.eval()
    embeddings, style_labels = [], []
    for images, s_lbls, _, _ in tqdm(loader, leave=False, desc="embed"):
        emb = model.get_embedding(images.to(device))
        embeddings.append(emb.cpu().numpy())
        style_labels.extend(s_lbls.numpy())
    return np.vstack(embeddings), np.array(style_labels)


embeddings, style_labels_arr = extract_embeddings(model, test_loader, DEVICE)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
from numpy.linalg import norm

def compute_outlier_scores(embeddings, labels):
    """Return cosine distance from each sample to its class centroid."""
    centroids = {}
    for cls in np.unique(labels):
        mask = labels == cls
        centroids[cls] = embeddings[mask].mean(axis=0)

    scores = np.zeros(len(labels))
    for i, (emb, lbl) in enumerate(zip(embeddings, labels)):
        c = centroids[lbl]
        cos_sim = np.dot(emb, c) / (norm(emb) * norm(c) + 1e-8)
        scores[i] = 1.0 - cos_sim   # distance (higher = more outlier)
    return scores


outlier_scores = compute_outlier_scores(embeddings, style_labels_arr)

# Top-10 outlier images
top_outlier_idx = np.argsort(outlier_scores)[::-1][:10]
outlier_df = test_df.iloc[top_outlier_idx].copy()
outlier_df["outlier_score"] = outlier_scores[top_outlier_idx]
print("Top-10 outlier paintings:")
outlier_df[[IMG_COL, STYLE_COL, "outlier_score"]]

In [ ]:
# ── t-SNE visualisation of embedding space ────────────────────────────────────
print("Running t-SNE (this may take a moment)...")
tsne = TSNE(n_components=2, perplexity=min(30, len(embeddings) - 1),
            random_state=SEED, n_iter=1000)
emb_2d = tsne.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1],
                     c=style_labels_arr, cmap="tab20",
                     alpha=0.7, s=20)
# Highlight outliers
ax.scatter(emb_2d[top_outlier_idx, 0], emb_2d[top_outlier_idx, 1],
           facecolors="none", edgecolors="red", s=100, linewidths=1.5,
           label="Top outliers")
plt.colorbar(scatter, ax=ax, label="Style class index")
ax.set_title("t-SNE of CNN-LSTM Embeddings (Style classes)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 10. Backbone Fine-Tuning (Stage 2)

After the LSTM head has converged, unfreeze the CNN backbone for end-to-end fine-tuning at a lower learning rate.

In [ ]:
# ── Unfreeze backbone ─────────────────────────────────────────────────────────
for p in model.cnn.parameters():
    p.requires_grad = True

# Lower LR for backbone, normal LR for new layers
optimizer_ft = optim.AdamW([
    {"params": model.cnn.parameters(),   "lr": LR * 0.1},
    {"params": model.lstm.parameters(),  "lr": LR},
    {"params": model.attention.parameters(), "lr": LR},
    {"params": model.style_head.parameters(),  "lr": LR},
    {"params": model.artist_head.parameters(), "lr": LR},
    {"params": model.genre_head.parameters(),  "lr": LR},
    {"params": criterion.parameters(),   "lr": LR},
], weight_decay=WEIGHT_DECAY)

scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_ft, T_max=10, eta_min=1e-7
)

print("Stage-2 fine-tuning optimizer configured.")
print(f"Total trainable params now: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

---
## 11. Summary & Discussion

### Architecture choices
| Design decision | Rationale |
|---|---|
| EfficientNet-B3 backbone | Good accuracy/efficiency trade-off; widely used for art analysis |
| Bi-LSTM over spatial patches | Captures horizontal AND vertical context across image regions |
| Attention pooling | Lets the model focus on the most discriminative patches |
| Multi-task learning | Shared representation improves all three tasks; related attributes regularise each other |
| Uncertainty-weighted loss | Handles different difficulty levels across style / artist / genre |
| Two-stage training | Protects pre-trained weights early; fine-tunes for domain adaptation |

### Evaluation metrics
| Metric | Why relevant |
|---|---|
| **Top-1 / Top-5 Accuracy** | Standard benchmark — matches ArtGAN paper |
| **Macro F1-Score** | Robust to class imbalance (very skewed artist distribution) |
| **Confusion Matrix** | Reveals systematic confusions (e.g. Impressionism ↔ Post-Impressionism) |
| **Cosine Outlier Score** | Pinpoints unusual or misattributed paintings |
| **t-SNE / UMAP** | Qualitative cluster analysis |

### Expected performance (WikiArt full dataset)
- Style accuracy: ~75–80 % (competitive with state-of-the-art CNN baselines)
- Artist accuracy: ~60–70 % (harder due to 195 artists with few examples each)
- Genre accuracy: ~80–85 % (fewer classes, cleaner signal)